# Bab 15. scikit-learn: Melatih dan Menilai dengan Jujur

Kode pendamping buku *Python untuk Machine Learning dan Data
Science*. Jalankan selnya berurutan dari atas, sebab sebagian
sel memakai peubah dari sel sebelumnya.

Notebook ini dibangkitkan dari naskah buku. Jangan disunting di
sini, sunting listing pada berkas `.tex` lalu bangkitkan ulang.

## Persiapan

Bab ini melanjutkan contoh dari bab sebelumnya. Jalankan sel ini lebih dahulu supaya datanya tersedia.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import (LinearRegression,
    LogisticRegression, Ridge)
from sklearn.model_selection import (KFold, StratifiedKFold,
    train_test_split, cross_val_score, GridSearchCV,
    TimeSeriesSplit)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, roc_auc_score,
    f1_score)

from siapkan import data_rumah

luas, kamar, y = data_rumah()

## 1. Membandingkan dengan Bab~bab:gd

In [ ]:
from sklearn.linear_model import LinearRegression

X = np.column_stack([luas, kamar])
m = LinearRegression().fit(X, y)

print(round(m.intercept_, 4))
print(np.round(m.coef_, 4))
print(round(m.score(X, y), 6))

Keluaran yang diharapkan:

```
142.9675
[ 4.4539 21.0554]
0.997907
```

## 2. Pembagian latih-uji

In [ ]:
from sklearn.model_selection import train_test_split

X_lat, X_uji, y_lat, y_uji = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

## 3. Validasi silang lima lipatan

In [ ]:
from sklearn.model_selection import (cross_val_score,
                                     StratifiedKFold)

cv = StratifiedKFold(5, shuffle=True, random_state=0)
skor = cross_val_score(model, X, y, cv=cv)

print(f"{skor.mean():.3f} +- {skor.std():.3f}")

Keluaran yang diharapkan:

```
0.745 +- 0.073
```

## 4. Kebocoran lewat seleksi fitur

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(60, 2000))
y = rng.integers(0, 2, 60)          # label acak murni

# SALAH: seleksi memakai seluruh data
sel = SelectKBest(f_classif, k=20).fit(X, y)
salah = cross_val_score(LogisticRegression(max_iter=2000),
                        sel.transform(X), y, cv=cv)

# BENAR: seleksi berada di dalam pipeline
pipe = Pipeline([
    ("sel", SelectKBest(f_classif, k=20)),
    ("clf", LogisticRegression(max_iter=2000)),
])
benar = cross_val_score(pipe, X, y, cv=cv)

print(f"{salah.mean():.3f}  {benar.mean():.3f}")

Keluaran yang diharapkan:

```
0.950  0.483
```

## 5. Pipeline sebagai pengaman

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipa = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000),
)

skor = cross_val_score(pipa, X, y, cv=cv)

## 6. Model yang tidak belajar apa-apa

In [ ]:
from sklearn.dummy import DummyClassifier

dum = DummyClassifier(strategy="most_frequent")
dum.fit(X_lat, y_lat)
print(accuracy_score(y_uji, dum.predict(X_uji)))
print(recall_score(y_uji, dum.predict(X_uji)))

Keluaran yang diharapkan:

```
0.943
0.000
```

## 7. Memilih ambang lewat kurva P--R

In [ ]:
prob = clf.predict_proba(X_uji)[:, 1]
pr, rc, th = precision_recall_curve(y_uji, prob)

f1 = 2*pr*rc / (pr + rc + 1e-12)
i = f1.argmax()
print(f"{th[i]:.3f} {f1[i]:.3f} {pr[i]:.3f} {rc[i]:.3f}")

Keluaran yang diharapkan:

```
0.187 0.486 0.450 0.529
```

## 8. Pencarian kisi

In [ ]:
from sklearn.model_selection import GridSearchCV

pipa = make_pipeline(StandardScaler(), Ridge())
kisi = {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]}

gs = GridSearchCV(pipa, kisi, cv=KFold(5, shuffle=True,
                  random_state=0), scoring="r2").fit(X, y)
print(gs.best_params_, round(gs.best_score_, 4))

Keluaran yang diharapkan:

```
{'ridge__alpha': 1} 0.9267
```

## 9. Validasi silang bersarang

In [ ]:
dalam = KFold(5, shuffle=True, random_state=0)
luar = KFold(5, shuffle=True, random_state=1)

gs = GridSearchCV(pipa, kisi, cv=dalam, scoring="r2")
nested = cross_val_score(gs, X, y, cv=luar, scoring="r2")